In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/q22_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

def_codes = ["13", "31", "23", "29", "30", "18", "17"]

# 1. extract country code and initial filters
customer_filtered = (
    customer[["C_CUSTKEY", "C_ACCTBAL", "C_PHONE"]]
    .assign(CNTRYCODE=customer.C_PHONE.str.slice(0, 2))
)
customer_filtered = customer_filtered[
    (customer_filtered.C_ACCTBAL > 0.0)
    & (customer_filtered.CNTRYCODE.isin(def_codes))
]

# 2. filter above-average balances
avg_balance = customer_filtered.C_ACCTBAL.mean()
customer_filtered = customer_filtered[customer_filtered.C_ACCTBAL > avg_balance]

# 3. anti-join: drop customers with any orders
customer_filtered = customer_filtered[~customer_filtered.C_CUSTKEY.isin(orders.O_CUSTKEY)]

# 4. single groupby for both aggregations
total = (
    customer_filtered
    .groupby("CNTRYCODE")
    .agg({"C_CUSTKEY": "count", "C_ACCTBAL": "sum"})
    .reset_index()
    .rename(columns={"C_CUSTKEY": "NUMCUST", "C_ACCTBAL": "TOTACCTBAL"})
    .sort_values("CNTRYCODE")
)
